In [1]:
import vec2text
import torch
import pickle
import numpy as np
from tqdm import  tqdm

def cosine_similarity(X, Y):
    """
    Calculates cosine similarity between corresponding rows of two matrices.

    Args:
        X (numpy.ndarray): First matrix.
        Y (numpy.ndarray): Second matrix.

    Returns:
        numpy.ndarray: Array of cosine similarities for each row pair.
    """

    return (X * Y).sum(axis=1) / np.linalg.norm(X, axis=1) / np.linalg.norm(Y, axis=1)

from transformers import AutoModel, AutoTokenizer, PreTrainedTokenizer, PreTrainedModel

/gpfs/radev/home/cc3385/.conda/envs/vec2text/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
encoder = AutoModel.from_pretrained("sentence-transformers/gtr-t5-base").encoder.to("cuda")
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/gtr-t5-base")
corrector = vec2text.load_pretrained_corrector("gtr-base")

def get_gtr_embeddings(text_list,
                       encoder: PreTrainedModel,
                       tokenizer: PreTrainedTokenizer) -> torch.Tensor:

    inputs = tokenizer(text_list,
                       return_tensors="pt",
                       max_length=128, 
                       truncation=True,
                       padding="max_length",).to("cuda")

    with torch.no_grad():
        model_output = encoder(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        hidden_state = model_output.last_hidden_state
        embeddings = vec2text.models.model_utils.mean_pool(hidden_state, inputs['attention_mask'])

    return embeddings

Loading checkpoint shards: 100%|██████████| 6/6 [00:02<00:00,  2.27it/s]
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [3]:
with open("../../data/pubmed_rct/processed_dataset/test_with_embeddings.pkl", 'rb') as f:
    test_dict = pickle.load(f)
test_abstracts = test_dict["abstracts"]
test_sections = test_dict["sections"]
test_section_texts = test_dict["section_texts"]

# Invert Embedding into Abstract

In [4]:
# create batches
batch_size = 64
batch_abstracts = [test_abstracts[n:n+batch_size] for n in range(0, len(test_abstracts), batch_size)]

decoded_abstract_list = []
cossim_list = []

pbar = tqdm(total=len(batch_abstracts))

for _, abstracts in enumerate(batch_abstracts):
    embeddings = get_gtr_embeddings(abstracts, encoder, tokenizer)
    
    decoded_abstracts = vec2text.invert_embeddings(
        embeddings=embeddings.cuda(),
        corrector=corrector,
        num_steps=20
    )

    decoded_embeddings = get_gtr_embeddings(decoded_abstracts, encoder, tokenizer)
    
    cossim = cosine_similarity(embeddings.cpu(), decoded_embeddings.cpu())

    decoded_abstract_list += decoded_abstracts
    cossim_list += cossim

    pbar.update(1)

pbar.close()

  0%|          | 0/40 [00:00<?, ?it/s]/tmp/tmp.ghpZWWKlH4/ipykernel_2989988/2802575976.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return (X * Y).sum(axis=1) / np.linalg.norm(X, axis=1) / np.linalg.norm(Y, axis=1)
100%|██████████| 40/40 [12:36<00:00, 18.92s/it]


In [5]:
np.mean(cossim_list), np.std(cossim_list)

(np.float32(0.6961544), np.float32(0.08083091))

In [27]:
len(cossim_list)

2500

In [16]:
decoded_abstract_list[:15]

['association with the MAP44 inhibitor may result in a proliferative complement injury (Bristol, S., et al.',
 'adolescents may help with a randomized controlled group (RCD) self-depressive symptoms decrease from a 5-year cohort with a 5-year',
 'in the prevention of nausea-causing tropislatin in the 5-estimated dose of 132,527-137,527-cis',
 'sed may decrease VO duration in adolescents who are not receiving adequate mechanical intervention. (Season, 2004; pp. 115–',
 'stimulation of one muscle to assess whether acupuncture function could be optimal for patients suffering systemic dysarnism. A study of 4,7',
 'randomized to withdraw patients in order to evaluate the predictive characteristics of hypertension. A comparative analysis of 310,536 - 310,',
 'a single 2.4-dose dose of oral telvery was found to be favourable for young adults aged between 0–60 years and up',
 'canolfazin is the 2-dose randomized controlled trial to provide a 97-to-1 (97-to-1)-s',
 'vitamin D deficiency is one 

In [17]:
with open('gtr-50steps-decoded-abstracts.pkl', 'wb') as file:
    pickle.dump(decoded_abstract_list, file)

# Pipeline 2 Abstract
section->emb->vec2text->decodedSection->concatenate

In [4]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from data_helpers import parse_articles_from_file

test_articles_obj = parse_articles_from_file("../../data/pubmed_rct/test.cln.txt")

In [5]:
reference_abstract_list_via_piepline = []
decoded_abstract_list_via_pipe = []

pbar = tqdm(total=len(test_articles_obj))

for article_id, article_dict in test_articles_obj.items():
    
    section_texts = list(article_dict.values())
    reference_abstract_list_via_piepline.append(" ".join(section_texts)) # keep for reference

    embeddings = get_gtr_embeddings(section_texts, encoder, tokenizer) 
    
    decoded_texts = vec2text.invert_embeddings(
        embeddings=embeddings.cuda(),
        corrector=corrector,
        num_steps=20,
    )
    
    decoded_abstract_list_via_pipe.append(" ".join(decoded_texts))

    pbar.update(1)
    
pbar.close()

100%|██████████| 2500/2500 [2:32:53<00:00,  3.67s/it]  


In [6]:
with open('gtr-20steps-pipeline2abstracts.pkl', 'wb') as file:
    pickle.dump((reference_abstract_list_via_piepline, decoded_abstract_list_via_pipe), file)

In [7]:
ref_embs = get_gtr_embeddings(reference_abstract_list_via_piepline, encoder, tokenizer)

In [8]:
eval_embs = get_gtr_embeddings(decoded_abstract_list_via_pipe, encoder, tokenizer)

In [13]:
cossim_list = cosine_similarity(ref_embs.cpu(), eval_embs.cpu())

/tmp/tmp.C2kCpjMDWX/ipykernel_3083269/2802575976.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return (X * Y).sum(axis=1) / np.linalg.norm(X, axis=1) / np.linalg.norm(Y, axis=1)


In [24]:
torch.mean(cossim_list), torch.std(cossim_list)

(tensor(0.8209), tensor(0.0693))

# Emb2Section

In [12]:
from collections import defaultdict

test_section_data_dict = defaultdict(lambda: defaultdict(list))
for section, text in zip(test_sections, test_section_texts):
    test_section_data_dict[section.lower()]["texts"].append(text)

In [43]:
decoded_results = defaultdict(list)

# evaluate each section
for key in test_section_data_dict.keys():
    
    list_of_cos = []
    target_section = key
    number_of_cases = len(test_section_data_dict[target_section]["texts"])

    for i in range(0, number_of_cases, batch_size): 
        embeddings = get_gtr_embeddings(test_section_data_dict[target_section]["texts"][i:i+batch_size], encoder, tokenizer) 
        
        decoded_texts = vec2text.invert_embeddings(
            embeddings=embeddings.cuda(),
            corrector=corrector,
            num_steps=20,
            sequence_beam_width=4
        )
        
        decoded_results[target_section] += decoded_texts

        decoded_embeddings = get_gtr_embeddings(decoded_texts, encoder, tokenizer)
        cossim = cosine_similarity(embeddings.cpu(), decoded_embeddings.cpu())
        list_of_cos += cossim
        
    print(f"{target_section}: {len(list_of_cos)} samples, Mean: {np.mean(list_of_cos):.4f}, Std: {np.std(list_of_cos):.4f}")

/tmp/tmp.tJa8FRvrHl/ipykernel_713909/2802575976.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return (X * Y).sum(axis=1) / np.linalg.norm(X, axis=1) / np.linalg.norm(Y, axis=1)


methods: 613 samples, Mean: 0.7595, Std: 0.0821
objective: 418 samples, Mean: 0.9077, Std: 0.0850
background: 260 samples, Mean: 0.8708, Std: 0.0836
results: 603 samples, Mean: 0.7722, Std: 0.0858
conclusions: 606 samples, Mean: 0.9008, Std: 0.0818


In [15]:
test_section_data_dict["conclusions"]["texts"][40:55]

['Intravenous infusion of ET-1 in healthy humans discretely increases diastolic blood pressure and profoundly decreases renal haemodynamics and excretion of sodium and water. Pretreatment with the calcium-channel blocking agent isradipine for 1 week in a clinically relevant dose does not interfere with the action of ET-1.',
 'This study shows that minimal flap tension does not influence recession reduction after 3 months when shallow recessions are treated by means of CAF. In the test group (with tension), the statistical analysis suggests that the higher the flap tension, the lower the recession reduction.',
 'Among cocaine-dependent homeless persons with psychiatric comorbidity undergoing behavioral addiction treatment, a reduction in comorbid psychiatric disorder prevalence was observed over 6 months. Not all participants improved, suggesting that even evidence-based addiction treatment will prove insufficient for a meaningful proportion of the dually diagnosed homeless population.'

In [16]:
decoded_results["conclusions"][40:55]

['intravenous injection of ET-1 decreases diastolic pressure and may be done in conjunction with renal physiology for up to four months',
 'whether flap tension reduces the recession after three months, when shallow recessions are treated with lower CAF. However, the study did not consider flap tension',
 'in cocaine-dependent homeless patients. Although treatment may have reduced the prevalence of psychiatric duocomorbidity, data continues to be',
 'available in an experimental toothpaste, and the use of place-whitening has been shown to be beneficial in reducing stain accumulation. In comparison to experimental non',
 'A therapy of slow platelet transfusion might be more effective for the prevention of platelet loss. Further studies will be required to strengthen this hypothesis. ',
 'ablation contributes to pain reduction via thermal slopping. Surgical COLAS has been proposed as an alternative method of recovery in trainee',
 'Dextrose prolotherapy was clinically effective and safe 

In [ ]:
with open('gtr-20steps-sbeam-decoded-sections.pkl', 'wb') as file:
    pickle.dump(decoded_results, file)

In [13]:
with open('gtr-20steps-sbeam-decoded-sections.pkl', 'rb') as file:
    decoded_results = pickle.load(file)

In [15]:
for key in test_section_data_dict.keys():
    ref_embs = embedding_model.encode(test_section_data_dict[key]["texts"])
    ev_embs = embedding_model.encode(decoded_results[key])
    sc_list = cosine_similarity(ref_embs, ev_embs)
    print(key, len(sc_list), np.mean(sc_list), np.std(sc_list))

methods 613 0.76659125 0.072993234
objective 418 0.8971013 0.089001924
background 260 0.86106026 0.081048325
results 603 0.76769483 0.08607027
conclusions 606 0.89176893 0.08004761
